# Test `colab_utils.py` trên Google Colab

Notebook này kiểm tra từng bước sau khi bạn upload `colab_utils.py`:
1. Import module.
2. Kiểm tra/cài Tailscale.
3. Kết nối Tailscale bằng Colab Secret.
4. Kiểm tra truy cập MLflow.
5. (Tùy chọn) Tìm và tải một model đã train.

> **Trước khi chạy:** upload `colab_utils.py` vào phiên Colab hiện tại và tạo Colab Secret tên `TAILSCALE_AUTH_KEY`. Không dán auth key trực tiếp vào notebook.

In [63]:
# Cell 1 — Upload file util từ máy tính
from google.colab import files

uploaded = files.upload()  # Chọn colab_utils.py
print("Uploaded:", list(uploaded.keys()))
import importlib
import colab_utils

importlib.invalidate_caches()
importlib.reload(colab_utils)

print("File đang dùng:", colab_utils.__file__)

Saving colab_utils.py to colab_utils (1).py
Uploaded: ['colab_utils (1).py']
File đang dùng: /content/colab_utils.py
Có start_run: True


In [52]:
# Cell 2 — Import module
import sys
sys.path.insert(0, "/content")

import colab_utils

print("Import OK:", colab_utils.__file__)
print("Có các hàm:", [
    name for name in [
        "install_tailscale",
        "setup_tailscale",
        "tailscale_is_connected",
        "setup_mlflow",
        "load_trained_model",
        "clear_model_cache",
        "download_kaggle_dataset",
    ]
    if hasattr(colab_utils, name)
])

Import OK: /content/colab_utils.py
Có các hàm: ['install_tailscale', 'setup_tailscale', 'tailscale_is_connected', 'setup_mlflow', 'load_trained_model', 'clear_model_cache']


In [ ]:
from colab_utils import (
    start_run,
    log_training_info,
    log_dataset_info,
    log_model,
    log_artifact,
    get_best_run,
    get_project_paths,
)

In [10]:
# Cell 3 — Cài Tailscale nếu runtime chưa có binary
# Chạy một lần cho mỗi runtime mới.
colab_utils.install_tailscale()
# cài mlflow
%pip install -q mlflow==3.16.1 pysocks kaggle

Đã tìm thấy tailscale và tailscaled.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 10.8 MB/s eta 0:00:00
   ━━

In [9]:
# Cell 4 — Kết nối Tailscale
# Cần tạo Colab Secret: TAILSCALE_AUTH_KEY
status = colab_utils.setup_tailscale()
print("Connected check:", colab_utils.tailscale_is_connected())

Tailscale có vẻ đã kết nối; bỏ qua khởi động lại.
Connected check: True


In [15]:
# Cell 5 — Cấu hình và kiểm tra MLflow
# Sửa server_uri nếu địa chỉ MLflow của bạn khác.
MLFLOW_URI = "http://100.116.34.36:8080"
EXPERIMENT_NAME = "Chest-Xray-CNN-LIME"

colab_utils.setup_mlflow(
    server_uri=MLFLOW_URI,
    experiment=EXPERIMENT_NAME,
)

MLflow server phản hồi HTTP 200: http://100.116.34.36:8080
MLflow tracking URI: http://100.116.34.36:8080
Experiment: Chest-Xray-CNN-LIME


'http://100.116.34.36:8080'

## Tùy chọn: test tải model

Chỉ chạy các cell dưới đây khi MLflow kết nối thành công và tên run/model đúng với dữ liệu của bạn.

In [16]:
# Cell 6 — Liệt kê run để kiểm tra tên model chính xác
import mlflow

mlflow.set_tracking_uri(MLFLOW_URI)
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=30,
    order_by=["start_time DESC"],
)

display(runs[[
    col for col in ["run_id", "tags.mlflow.runName", "status", "start_time"]
    if col in runs.columns
]])

,run_id,tags.mlflow.runName,status,start_time
0,38fac9402fb5423082ecbc6475e85958,resnet50,FINISHED,2026-10-09 04:24:49.346000+00:00
1,11b5247ea6b94cdbadc271e7b7cc13ac,resnet18_robust,FINISHED,2026-10-09 04:00:32.532000+00:00
2,20a84d72dd744d2aabc19d19fbff95b9,bn_cnn,FINISHED,2026-10-09 03:34:02.531000+00:00


In [17]:
# Cell 7 — Tải một model theo tên run
# Đổi MODEL_RUN_NAME thành tên chính xác trong cột tags.mlflow.runName.
MODEL_RUN_NAME = "resnet50"

model = colab_utils.load_trained_model(
    MODEL_RUN_NAME,
    experiment=EXPERIMENT_NAME,
)

print(type(model))
print("Training/eval mode:", model.training)  # False nghĩa là model.eval() đang bật

[resnet50] run 38fac940 -> model m-277ed65f217040f4b0982a92871f4506
[resnet50] tải MLmodel: 1,082 bytes (0.3s)
[resnet50] tải data/pickle_module_info.txt: 28 bytes (0.3s)


2026/10/10 06:24:10 WARNING mlflow.pytorch: Stored model version '2.11.0+cu130' does not match installed PyTorch version '2.11.0+cpu'


[resnet50] tải data/model.pth: 94,371,955 bytes (51.5s)
[resnet50] nạp OK: ResNet
<class 'torchvision.models.resnet.ResNet'>
Training/eval mode: False


In [18]:
# Cell 8 — Kiểm tra cache: gọi lại sẽ dùng model đã nạp trong runtime
model_again = colab_utils.load_trained_model(MODEL_RUN_NAME)
print("Cùng object trong cache:", model is model_again)

Cùng object trong cache: True


In [19]:
# Cell 9 — Khi Tailscale rớt kết nối, thử khôi phục
# Chỉ chạy khi cần; sau đó kiểm tra lại MLflow ở Cell 5.
colab_utils.setup_tailscale(force_reconnect=True)
print("Connected check:", colab_utils.tailscale_is_connected())

100.110.38.52  google-colab-1  userid:4671521171800777  linux  offline
HTTP proxy: http://localhost:1056
SOCKS5 proxy: socks5://localhost:1055
Connected check: True


## Data

## Cách khác

In [30]:
# tải lại từ đầu
DATA_DIR = download_kaggle_dataset(
    dataset="uciml/iris",
    data_dir="/content/data/iris",
    force=True,
)

Đang tải dataset: uciml/iris
Đang giải nén: iris.zip
Hoàn tất! Dữ liệu nằm tại: /content/data/iris


In [42]:

from colab_utils import (
    # setup_tailscale,
    # setup_mlflow,
    start_run,
    log_training_info,
    log_dataset_info,
    log_model,
    log_artifact,
    get_best_run,
    get_project_paths,
)

import mlflow
import torch

# 1. Chuẩn bị môi trường

paths = get_project_paths("/content/chest_xray_project")

# 2. Bắt đầu một run
with start_run(
    run_name="resnet50",
    tags={"model_type": "ResNet50", "task": "classification"},
):
    # 3. Ghi nhận dataset
    log_dataset_info(
        data_dir="/content/chest_xray_data/chest_xray",
        dataset_name="chest-xray-pneumonia",
        version="kaggle",
    )

    # 4. Log cấu hình huấn luyện
    log_training_info(
        params={
            "architecture": "ResNet50",
            "epochs": 5,
            "batch_size": 32,
            "learning_rate": 0.001,
            "optimizer": "Adam",
        }
    )

    # 5. Ví dụ: gọi vòng lặp huấn luyện của bạn tại đây
    # model, history = train_model(...)

    # 6. Sau mỗi epoch, log metrics thực tế
    # log_training_info(
    #     metrics={
    #         "train_loss": train_loss,
    #         "val_loss": val_loss,
    #         "val_accuracy": val_accuracy,
    #     },
    #     step=epoch,
    # )

    # 7. Sau khi huấn luyện, log model thật
    # log_model(model, artifact_path="resnet50")

    # 8. Log biểu đồ hoặc ảnh LIME
    # log_artifact("/content/explanation.png", "lime")

# 9. Tìm run tốt nhất
# best_run = get_best_run(metric="val_accuracy")


Dataset: chest-xray-pneumonia
Số file: 17,591
Dung lượng: 2368.85 MB
🏃 View run resnet50 at: http://100.116.34.36:8080/#/experiments/1/runs/46f53d5ea1674e368c20038e5bbc99a4
🧪 View experiment at: http://100.116.34.36:8080/#/experiments/1


In [65]:
with start_run(
    run_name="resnet50",
    experiment="Chest-Xray-CNN-LIME",
    tags={
        "model_type": "ResNet50",
        "task": "classification",
    },
):
    # Ghi nhận dataset
    log_dataset_info(
        data_dir="/content/chest_xray_data/chest_xray",
        dataset_name="chest-xray-pneumonia",
        version="kaggle",
    )

    # Ghi cấu hình huấn luyện
    log_training_info(
        params={
            "architecture": "ResNet50",
            "epochs": 5,
            "batch_size": 32,
            "learning_rate": 0.001,
            "optimizer": "Adam",
        }
    )

    # TODO: Huấn luyện model tại đây.
    # Sau mỗi epoch, log metrics thực tế.
    # Sau khi huấn luyện, log model và artifact nếu cần.

Dataset: chest-xray-pneumonia
Số file: 17,591
Dung lượng: 2368.85 MB
🏃 View run resnet50 at: http://100.116.34.36:8080/#/experiments/2/runs/945365ac70fe4dfc9678037bf47f5bcf
🧪 View experiment at: http://100.116.34.36:8080/#/experiments/2


In [45]:
import mlflow
from IPython.display import display

runs = mlflow.search_runs(
    experiment_names=["Chest-Xray-CNN-LIME"],
    max_results=10,
)

print("Các cột hiện có:", runs.columns.tolist())

if runs.empty:
    print("Chưa có run trong experiment này.")
else:
    # Chỉ chọn các cột thực sự tồn tại
    columns_to_show = [
        col for col in [
            "run_id",
            "run_uuid",
            "tags.mlflow.runName",
            "status",
            "start_time",
            "metrics.val_accuracy",
        ]
        if col in runs.columns
    ]

    display(runs[columns_to_show])

Các cột hiện có: ['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'metrics.test_auc', 'metrics.test05_threshold', 'metrics.test05_recall', 'metrics.val_acc', 'metrics.testthr_threshold', 'metrics.test05_FN', 'metrics.test05_precision', 'metrics.train_acc', 'metrics.testthr_specificity', 'metrics.val_loss', 'metrics.val_recall', 'metrics.val_precision', 'metrics.testthr_recall', 'metrics.test05_specificity', 'metrics.testthr_precision', 'metrics.testthr_FN', 'metrics.testthr_accuracy', 'metrics.train_loss', 'metrics.test05_accuracy', 'metrics.test05_FP', 'metrics.testthr_FP', 'metrics.test_ap', 'metrics.val_threshold', 'params.optimizer', 'params.batch_size', 'params.architecture', 'params.learning_rate', 'params.epochs', 'params.tag', 'params.split_seed', 'params.model', 'params.patience', 'params.augmentation', 'tags.dataset.dataset_name', 'tags.dataset.dataset_size_bytes', 'tags.dataset.dataset_path', 'tags.mlflow.runName', 'tags.mlflow.source.name', 't

,run_id,tags.mlflow.runName,status,start_time
0,54485ba3522f47f3a3e40743bb785286,resnet50,FINISHED,2026-10-10 06:59:51.799000+00:00
1,38fac9402fb5423082ecbc6475e85958,resnet50,FINISHED,2026-10-09 04:24:49.346000+00:00
2,11b5247ea6b94cdbadc271e7b7cc13ac,resnet18_robust,FINISHED,2026-10-09 04:00:32.532000+00:00
3,20a84d72dd744d2aabc19d19fbff95b9,bn_cnn,FINISHED,2026-10-09 03:34:02.531000+00:00


In [56]:
import sys
sys.path.insert(0, "/content")

import colab_utils

print("File:", colab_utils.__file__)

print([
    name for name in dir(colab_utils)
    if not name.startswith("_") and callable(getattr(colab_utils, name))
])

File: /content/colab_utils.py
['MLflowModelLoader', 'Optional', 'Path', 'clear_model_cache', 'get_secret', 'install_tailscale', 'load_trained_model', 'setup_mlflow', 'setup_tailscale', 'tailscale_is_connected']
